In [ ]:
# import essential
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [ ]:
# collect units location information
from general_utils import find_ephys_sessions
from pathlib import Path
from ephys_utils import collect_units_locations_to_csv

# Step 1: discover sessions
_, _, sessions = find_ephys_sessions()
print(f"Total sessions found: {len(sessions)}")


# Step 2: export unit locations
OUT_CSV = Path("/root/capsule/scratch/unit_locations_by_session.csv")

df_unitloc = collect_units_locations_to_csv(
    sessions,
    out_csv=OUT_CSV,
    verbose=True,
)

print("Done.")


In [8]:
# summary units by region
from general_utils import load_df_with_psth_csv
from behavior_qc_visualization import load_behavior_model_summary_csv
from behavior_qc_metrics_summary import append_model_criteria_result
from behavior_qc_metrics_summary import append_summary_to_df_by_session

# load the unit_locations
df_unitloc=load_df_with_psth_csv("/root/capsule/scratch/unit_locations_by_session.csv")


# Summary
summary = load_behavior_model_summary_csv("/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv")
summary = append_model_criteria_result(summary)

# append the qc
df_with_summary = append_summary_to_df_by_session(
    df_unitloc,
    summary,
    df_session_col="session_name",
    summary_session_col="session_name",
    summary_cols=[
        "QLearning_L1F1_CK1_softmax_pass_all_criteria",
    ],
    prefix="summary_",
)


[LOAD] Reading CSV: /root/capsule/scratch/unit_locations_by_session.csv
[LOAD] Restoring 0 array columns...
[LOAD] Done.


In [19]:
import pandas as pd


def summarize_units_by_region_and_session(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize unit statistics per session (one row per session).

    Output columns:
      - total_units, total_pass, total_fail, fraction_pass
      - n_units_<region> for each brain_region
      - n_pass_<region> for each brain_region
      - fraction_pass_<region> for each brain_region
    """
    pass_col = "summary_QLearning_L1F1_CK1_softmax_pass_all_criteria"
    df = df.copy()

    # Normalize brain_region / pass flag
    df["brain_region"] = df["brain_region"].fillna("Unknown").astype(str)
    df[pass_col] = df[pass_col].fillna(False).astype(bool)

    # -----------------------------
    # Overall session-level summary
    # -----------------------------
    overall = (
        df.groupby("session_name")
        .agg(
            total_units=("unit_id", "count"),
            total_pass=(pass_col, "sum"),  # bool sum counts True
        )
        .astype({"total_units": int, "total_pass": int})
    )
    overall["total_fail"] = overall["total_units"] - overall["total_pass"]
    overall["fraction_pass"] = overall["total_pass"] / overall["total_units"]

    # -----------------------------
    # Per-region counts (units)
    # -----------------------------
    units_counts = df.groupby(["session_name", "brain_region"]).size().unstack(fill_value=0)

    # Per-region counts (pass)
    pass_counts = (
        df[df[pass_col]]
        .groupby(["session_name", "brain_region"])
        .size()
        .unstack(fill_value=0)
    )

    # Align both to same index/columns
    all_sessions = units_counts.index.union(pass_counts.index)
    all_regions = units_counts.columns.union(pass_counts.columns)

    units_counts = units_counts.reindex(index=all_sessions, columns=all_regions, fill_value=0)
    pass_counts = pass_counts.reindex(index=all_sessions, columns=all_regions, fill_value=0)

    # Fractions per region (avoid divide-by-zero)
    fraction_counts = pass_counts.divide(units_counts.where(units_counts != 0), fill_value=0)

    # Add prefixes
    units_pivot = units_counts.add_prefix("n_units_")
    pass_pivot = pass_counts.add_prefix("n_pass_")
    fraction_pivot = fraction_counts.add_prefix("fraction_pass_")

    # Combine
    out = pd.concat([overall, units_pivot, pass_pivot, fraction_pivot], axis=1).reset_index()

    return out


In [20]:
summarize_units_by_region_and_session(df_with_summary)

/tmp/ipykernel_2071155/2521116575.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[pass_col] = df[pass_col].fillna(False).astype(bool)


,session_name,total_units,total_pass,total_fail,fraction_pass,n_units_AAA,n_units_ACAd6a,n_units_ACAv1,n_units_ACAv2/3,n_units_ACAv5,...,fraction_pass_Unknown,fraction_pass_V3,fraction_pass_VM,fraction_pass_VPMpc,fraction_pass_ccg,fraction_pass_fiber tracts,fraction_pass_fx,fraction_pass_opt,fraction_pass_root,fraction_pass_void
0,ecephys_753124_2024-12-10_17-24-56_sorted_2024...,1972,0,1972,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ecephys_753125_2024-10-09_10-50-19_sorted_2024...,1382,0,1382,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ecephys_753125_2024-10-10_14-41-23_sorted_2024...,643,0,643,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ecephys_753125_2024-10-14_15-37-15_sorted_2024...,1143,0,1143,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ecephys_753125_2024-10-15_16-16-22_sorted_2024...,967,0,967,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ecephys_753126_2024-10-10_17-51-24_sorted_2025...,1437,0,1437,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ecephys_753126_2024-10-11_13-14-24_sorted_2024...,970,0,970,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ecephys_753126_2024-10-15_12-20-35_sorted_2024...,1070,0,1070,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ecephys_764769_2024-12-11_18-21-49_sorted_2024...,1592,1592,0,1.0,0,0,0,0,0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ecephys_764769_2024-12-12_16-05-00_sorted_2024...,1051,0,1051,0.0,0,0,0,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
